# extlib

> External library indexing for documentation lookup

In [ ]:
#| default_exp utils.extlib

In [ ]:
#| hide
from nbdev.showdoc import *

## Imports

In [ ]:
#| export
from __future__ import annotations
from typing import Any, Dict, List, Optional, Set, Tuple
from dataclasses import dataclass, field
from pathlib import Path
import os
import sys
import json
import importlib
import importlib.util
import inspect
import pkgutil

## Constants

In [ ]:
#| export
# Default resources directory
PACKAGE_RESOURCES = Path(__file__).parent.parent / 'resources' if '__file__' in dir() else Path.cwd() / 'resources'
EXTERNAL_LIBS_DIR = PACKAGE_RESOURCES / 'external_libs'

# Known documentation URLs for popular packages
KNOWN_DOC_URLS: Dict[str, str] = {
    'torch': 'https://pytorch.org/docs/stable',
    'numpy': 'https://numpy.org/doc/stable/reference',
    'pandas': 'https://pandas.pydata.org/docs/reference',
    'scipy': 'https://docs.scipy.org/doc/scipy/reference',
    'sklearn': 'https://scikit-learn.org/stable/modules/generated',
    'transformers': 'https://huggingface.co/docs/transformers/main/en',
    'tensorflow': 'https://www.tensorflow.org/api_docs/python',
    'keras': 'https://keras.io/api',
    'matplotlib': 'https://matplotlib.org/stable/api',
    'fastai': 'https://docs.fast.ai',
    'fastcore': 'https://fastcore.fast.ai',
    'nbdev': 'https://nbdev.fast.ai',
}

## `NbdevLookup` Class

Provides documentation lookup for a package.

In [ ]:
#| export
@dataclass
class NbdevLookup:
    """Documentation lookup for a Python package.
    
    Provides symbol-to-documentation-URL mapping similar to nbdev's
    built-in lookup functionality.
    
    Parameters
    ----------
    modidx : Dict[str, Any]
        The module index dictionary (from _modidx.py 'd' variable).
    
    Attributes
    ----------
    syms : Dict[str, Tuple[str, str]]
        Mapping of symbol names to (doc_url, source_file) tuples.
    """
    modidx: Dict[str, Any] = field(default_factory=dict)
    
    def __post_init__(self):
        """Build the symbols lookup table."""
        self.syms: Dict[str, Tuple[str, str]] = {}
        syms_dict = self.modidx.get('syms', {})
        mods_dict = self.modidx.get('mods', {})
        
        # Flatten syms from all modules
        for mod_name, mod_syms in syms_dict.items():
            if isinstance(mod_syms, dict):
                for sym_name, sym_info in mod_syms.items():
                    self.syms[sym_name] = sym_info
        
        # Add module entries
        if isinstance(mods_dict, dict):
            self.syms.update(mods_dict)
    
    def doc_link(self, symbol: str) -> Optional[str]:
        """Get the documentation URL for a symbol.
        
        Parameters
        ----------
        symbol : str
            The fully qualified symbol name.
        
        Returns
        -------
        str or None
            The documentation URL, or None if not found.
        """
        info = self.syms.get(symbol)
        if info and isinstance(info, (list, tuple)) and len(info) > 0:
            return info[0]
        return None
    
    def source_file(self, symbol: str) -> Optional[str]:
        """Get the source file for a symbol.
        
        Parameters
        ----------
        symbol : str
            The fully qualified symbol name.
        
        Returns
        -------
        str or None
            The source file path, or None if not found.
        """
        info = self.syms.get(symbol)
        if info and isinstance(info, (list, tuple)) and len(info) > 1:
            return info[1]
        return None
    
    def search(self, query: str, limit: int = 10) -> List[str]:
        """Search for symbols matching a query.
        
        Parameters
        ----------
        query : str
            Search string (case-insensitive substring match).
        limit : int
            Maximum results to return.
        
        Returns
        -------
        List[str]
            List of matching symbol names.
        """
        q = query.lower()
        results = []
        for sym in self.syms:
            if q in sym.lower():
                results.append(sym)
                if len(results) >= limit:
                    break
        return results

## Index Generation

Functions for generating module indices for external packages.

In [ ]:
#| export
def get_package_modidx(package_name: str) -> Optional[Dict[str, Any]]:
    """Try to get an existing _modidx.py from an installed package.
    
    Parameters
    ----------
    package_name : str
        The package name to look up.
    
    Returns
    -------
    Dict[str, Any] or None
        The modidx 'd' dict if found, None otherwise.
    """
    try:
        # Try to import the package's _modidx
        modidx_mod = importlib.import_module(f'{package_name}._modidx')
        if hasattr(modidx_mod, 'd'):
            return modidx_mod.d
    except (ImportError, ModuleNotFoundError):
        pass
    return None

In [ ]:
#| export
def get_public_symbols(module) -> List[Tuple[str, Any]]:
    """Get public symbols from a module.
    
    Parameters
    ----------
    module
        The module to inspect.
    
    Returns
    -------
    List[Tuple[str, Any]]
        List of (name, object) tuples for public symbols.
    """
    symbols = []
    
    # Use __all__ if available
    if hasattr(module, '__all__'):
        names = module.__all__
    else:
        # Get non-private names
        names = [n for n in dir(module) if not n.startswith('_')]
    
    for name in names:
        try:
            obj = getattr(module, name)
            # Only include functions, classes, and modules
            if inspect.isfunction(obj) or inspect.isclass(obj) or inspect.ismodule(obj):
                symbols.append((name, obj))
        except Exception:
            pass
    
    return symbols

In [ ]:
#| export
def generate_modidx(
    package_name: str,
    doc_base_url: Optional[str] = None,
    max_depth: int = 2
) -> Dict[str, Any]:
    """Generate a module index for an installed package.
    
    Inspects the package to discover public symbols and creates
    a modidx-compatible dictionary.
    
    Parameters
    ----------
    package_name : str
        The package to index.
    doc_base_url : str, optional
        Base URL for documentation. Uses KNOWN_DOC_URLS if not specified.
    max_depth : int
        Maximum submodule depth to scan.
    
    Returns
    -------
    Dict[str, Any]
        A modidx-compatible dictionary.
    """
    base_url = doc_base_url or KNOWN_DOC_URLS.get(package_name, '')
    
    modidx = {
        'settings': {
            'lib_path': package_name,
            'doc_host': base_url,
        },
        'syms': {},
        'mods': {},
    }
    
    try:
        pkg = importlib.import_module(package_name)
    except ImportError:
        return modidx
    
    def process_module(mod, mod_name: str, depth: int = 0):
        if depth > max_depth:
            return
        
        mod_syms = {}
        
        for name, obj in get_public_symbols(mod):
            full_name = f'{mod_name}.{name}'
            
            # Generate doc URL
            if base_url:
                doc_url = f'{base_url}/{mod_name}.{name}.html'
            else:
                doc_url = ''
            
            # Get source file
            try:
                source_file = inspect.getfile(obj)
            except (TypeError, OSError):
                source_file = ''
            
            mod_syms[full_name] = (doc_url, source_file)
            
            # If it's a submodule, recurse
            if inspect.ismodule(obj) and hasattr(obj, '__name__'):
                if obj.__name__.startswith(package_name):
                    process_module(obj, obj.__name__, depth + 1)
        
        if mod_syms:
            modidx['syms'][mod_name] = mod_syms
        modidx['mods'][mod_name] = (f'{base_url}/{mod_name}.html' if base_url else '', '')
    
    process_module(pkg, package_name)
    
    # Also scan submodules
    if hasattr(pkg, '__path__'):
        for importer, modname, ispkg in pkgutil.walk_packages(
            pkg.__path__, prefix=f'{package_name}.'
        ):
            parts = modname.split('.')
            if len(parts) - 1 <= max_depth and not any(p.startswith('_') for p in parts[1:]):
                try:
                    submod = importlib.import_module(modname)
                    process_module(submod, modname, len(parts) - 1)
                except Exception:
                    pass
    
    return modidx

## Cache Management

Functions for caching external library indices.

In [ ]:
#| export
def get_cache_dir(package_name: str) -> Path:
    """Get the cache directory for an external package.
    
    Parameters
    ----------
    package_name : str
        The package name.
    
    Returns
    -------
    Path
        Path to the package's cache directory.
    """
    return EXTERNAL_LIBS_DIR / package_name

In [ ]:
#| export
def save_package_index(package_name: str, modidx: Dict[str, Any]) -> Path:
    """Save a package index to the cache.
    
    Creates the directory structure:
    - __init__.py (empty)
    - _modidx.py (the index)
    - _nbdev.py (lookup helper)
    
    Parameters
    ----------
    package_name : str
        The package name.
    modidx : Dict[str, Any]
        The module index dictionary.
    
    Returns
    -------
    Path
        Path to the created directory.
    """
    cache_dir = get_cache_dir(package_name)
    cache_dir.mkdir(parents=True, exist_ok=True)
    
    # Write __init__.py
    (cache_dir / '__init__.py').write_text('', encoding='utf-8')
    
    # Write _modidx.py
    modidx_content = f'# Auto-generated module index for {package_name}\n\nd = {repr(modidx)}\n'
    (cache_dir / '_modidx.py').write_text(modidx_content, encoding='utf-8')
    
    # Write _nbdev.py
    nbdev_content = f'''"""nbdev-style lookup for {package_name}."""

__all__ = ['NbdevLookup', 'modidx']

from . import _modidx
modidx = _modidx.d

class NbdevLookup:
    """Documentation lookup for {package_name}."""
    
    def __init__(self):
        py_syms = {{k: v for k_, v_ in modidx.get('syms', {{}}).items() for k, v in v_.items()}}
        self.syms = {{**py_syms, **modidx.get('mods', {{}})}}
    
    def doc_link(self, s: str):
        """Get documentation link for a symbol."""
        info = self.syms.get(s)
        if info and isinstance(info, (list, tuple)) and len(info) > 0:
            return info[0]
        return None
'''
    (cache_dir / '_nbdev.py').write_text(nbdev_content, encoding='utf-8')
    
    return cache_dir

In [ ]:
#| export
def load_package_index(package_name: str) -> Optional[NbdevLookup]:
    """Load a cached package index.
    
    Parameters
    ----------
    package_name : str
        The package name.
    
    Returns
    -------
    NbdevLookup or None
        The lookup object, or None if not cached.
    """
    cache_dir = get_cache_dir(package_name)
    modidx_path = cache_dir / '_modidx.py'
    
    if not modidx_path.exists():
        return None
    
    try:
        import runpy
        data = runpy.run_path(str(modidx_path))
        d = data.get('d')
        if isinstance(d, dict):
            return NbdevLookup(d)
    except Exception:
        pass
    
    return None

In [ ]:
#| export
def get_or_create_index(package_name: str, force: bool = False) -> NbdevLookup:
    """Get a package index, creating it if necessary.
    
    Parameters
    ----------
    package_name : str
        The package name.
    force : bool
        If True, regenerate even if cached.
    
    Returns
    -------
    NbdevLookup
        The lookup object for the package.
    """
    if not force:
        lookup = load_package_index(package_name)
        if lookup:
            return lookup
    
    # Try to get existing modidx from the package
    modidx = get_package_modidx(package_name)
    
    if not modidx:
        # Generate our own
        modidx = generate_modidx(package_name)
    
    # Cache it
    save_package_index(package_name, modidx)
    
    return NbdevLookup(modidx)

## Project Scanning

Functions for scanning a project's external dependencies.

In [ ]:
#| export
def scan_project_imports(project: Path) -> Set[str]:
    """Scan a project's notebooks for external imports.
    
    Parameters
    ----------
    project : Path
        Path to the nbdev project root.
    
    Returns
    -------
    Set[str]
        Set of external package names imported in the project.
    """
    from nbdev_mcp.utils.paths import iter_notebooks, read_nb, lib_name
    from nbdev_mcp.utils.nb import NBFile, extract_imports, split_imports
    
    lib = lib_name(project)
    external = set()
    
    for nb_path in iter_notebooks(project):
        try:
            data = read_nb(nb_path)
            nb = NBFile(nb_path, data)
            
            for cell in nb.code_cells():
                imports = extract_imports(cell.source)
                _, ext = split_imports(imports, lib)
                external.update(ext)
        except Exception:
            pass
    
    # Filter out standard library modules
    stdlib = set(sys.stdlib_module_names) if hasattr(sys, 'stdlib_module_names') else set()
    external -= stdlib
    external -= {'__future__', 'typing', 'typing_extensions'}
    
    return external

In [ ]:
#| export
def index_project_externals(
    project: Path,
    force: bool = False
) -> Dict[str, Any]:
    """Index all external libraries used by a project.
    
    Scans the project for imports and creates/updates indices
    for all external packages.
    
    Parameters
    ----------
    project : Path
        Path to the nbdev project root.
    force : bool
        If True, regenerate all indices.
    
    Returns
    -------
    Dict[str, Any]
        Result with 'packages', 'indexed', 'failed' lists.
    """
    external = scan_project_imports(project)
    indexed = []
    failed = []
    
    for pkg in sorted(external):
        try:
            get_or_create_index(pkg, force=force)
            indexed.append(pkg)
        except Exception as e:
            failed.append({'package': pkg, 'error': str(e)})
    
    return {
        'ok': True,
        'packages': list(external),
        'indexed': indexed,
        'failed': failed,
        'cache_dir': str(EXTERNAL_LIBS_DIR)
    }

In [ ]:
#| export
def lookup_symbol(symbol: str, packages: Optional[List[str]] = None) -> Optional[Dict[str, Any]]:
    """Look up a symbol across multiple package indices.
    
    Parameters
    ----------
    symbol : str
        The symbol to look up.
    packages : List[str], optional
        Packages to search. If None, searches all cached packages.
    
    Returns
    -------
    Dict[str, Any] or None
        Dict with 'package', 'doc_url', 'source_file' if found.
    """
    if packages is None:
        # List all cached packages
        if EXTERNAL_LIBS_DIR.exists():
            packages = [d.name for d in EXTERNAL_LIBS_DIR.iterdir() if d.is_dir()]
        else:
            packages = []
    
    for pkg in packages:
        lookup = load_package_index(pkg)
        if lookup:
            doc_url = lookup.doc_link(symbol)
            if doc_url:
                return {
                    'package': pkg,
                    'symbol': symbol,
                    'doc_url': doc_url,
                    'source_file': lookup.source_file(symbol)
                }
    
    return None

## Next

In [ ]:
#| export



## Export

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()